# ABCD ML Pipeline — Demo

Minimal end-to-end run of the stacking ensemble on synthetic data that matches the ABCD variable structure. Confirms the pipeline installs and executes without requiring NDA-licensed data.

**What this demo covers:**

- Loading the synthetic panel CSV (`CLEAN_ABCD_5_1_panel_synthetic.csv`)
- Feature/target selection at one timepoint (T0, baseline)
- Preprocessing fit on training partition only (median imputation + standardization)
- Stacking ensemble fit: CatBoost + XGBoost + Random Forest base learners, linear meta-learner
- Test-set evaluation and predicted-vs-observed plot

**What this demo does not cover** (see the main notebook `ABCDMLPipeline.ipynb` for these):

- TabPFN base learner (GPU-dependent; auto-excluded here for CPU-compatible execution)
- Circularity exclusion protocol, SHAP decomposition, conformal prediction intervals
- Nested CV, permutation testing, family-/site-grouped CV, domain ablation, temporal gradient

**Expected runtime:** ~1–2 minutes on a standard Colab runtime.

**Expected performance on synthetic data:** near-zero R². The synthetic CSV was generated by independent column-wise permutation, which breaks real feature–target associations. A runnable demo, not a reproduction of manuscript results.

In [ ]:
# Install dependencies (skip if pre-installed).
!pip install -q catboost xgboost scikit-learn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42

## 1. Load synthetic data

In [ ]:
DATA_PATH = 'CLEAN_ABCD_5_1_panel_synthetic.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Rows x columns : {df.shape[0]} x {df.shape[1]}")
print(f"Unique subjects: {df['subject'].nunique()}")
print(f"Timepoints     : {sorted(df['time'].unique())}")
print(f"Rows per wave  : {df['time'].value_counts().sort_index().to_dict()}")

## 2. Configure analysis

Predict the core depression composite (`sbt_core_depression`) at T0 from a compact feature set covering cognitive task performance, demographics, and genetic ancestry principal components. T0 is used because baseline cognitive data is densest in ABCD.

In [ ]:
TARGET = 'sbt_core_depression'
TIMEPOINT = 0

feature_cols = [
    'sex', 'parent_income',
    'cct',
    'tb_picvocab', 'tb_flanker', 'tb_list', 'tb_cardsort', 'tb_pattern',
    'tb_picture', 'tb_reading', 'tb_fluid', 'tb_cryst', 'tb_total',
    'ravlt_s_total',
] + [f'pc_gene_aces{i}' for i in range(1, 11)]

sub = df[df['time'] == TIMEPOINT].copy()
sub = sub.dropna(subset=[TARGET])

# Retain only features with > 50% coverage at this timepoint
feature_cols = [
    c for c in feature_cols
    if c in sub.columns and sub[c].notna().mean() > 0.5
]

X = sub[feature_cols]
y = sub[TARGET]

print(f"N rows  : {X.shape[0]}")
print(f"N feats : {X.shape[1]}")
print(f"Target  : {TARGET} (mean={y.mean():.3f}, sd={y.std():.3f})")

## 3. Train/test split and preprocessing

70/30 split, matching the main pipeline. Imputer and scaler are fit on training data only to avoid leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)

imputer = SimpleImputer(strategy='median').fit(X_train)
X_train_imp = imputer.transform(X_train)
X_test_imp  = imputer.transform(X_test)

scaler = StandardScaler().fit(X_train_imp)
X_train_s = scaler.transform(X_train_imp)
X_test_s  = scaler.transform(X_test_imp)

print(f"Train: {X_train_s.shape}")
print(f"Test : {X_test_s.shape}")

## 4. Fit the stacking ensemble

Base learners: CatBoost, XGBoost, Random Forest. Meta-learner: LinearRegression. The main pipeline also includes TabPFN; it is omitted here to keep the demo CPU-compatible.

In [ ]:
base_learners = [
    ('catboost', CatBoostRegressor(
        iterations=300, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, verbose=False, random_state=RANDOM_STATE,
    )),
    ('xgboost', XGBRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    )),
    ('random_forest', RandomForestRegressor(
        n_estimators=200, max_depth=10,
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
]

stack = StackingRegressor(
    estimators=base_learners,
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1,
)

print("Fitting stacking ensemble ...")
stack.fit(X_train_s, y_train)
print("Done.")

## 5. Evaluate

Negative or near-zero R² is expected on the synthetic data, which has no preserved feature–target associations. A positive train R² with near-zero test R² would indicate the demo is running correctly.

In [ ]:
y_pred_train = stack.predict(X_train_s)
y_pred_test  = stack.predict(X_test_s)

print(f"Training R^2 : {r2_score(y_train, y_pred_train):.3f}")
print(f"Test R^2     : {r2_score(y_test, y_pred_test):.3f}")
print(f"Test MAE     : {mean_absolute_error(y_test, y_pred_test):.3f}")
print(f"Test RMSE    : {np.sqrt(mean_squared_error(y_test, y_pred_test)):.3f}")

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(y_test, y_pred_test, alpha=0.5, s=22)
lims = [float(min(y_test.min(), y_pred_test.min())),
        float(max(y_test.max(), y_pred_test.max()))]
ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1)
ax.set_xlabel(f'Observed {TARGET}')
ax.set_ylabel(f'Predicted {TARGET}')
ax.set_title(f'Stacking ensemble, test set (n={len(y_test)})')
plt.tight_layout()
plt.show()

## Next steps

For the full pipeline (TabPFN, SHAP, Sankey visualizations, validation suite, cross-timepoint trajectories), open `ABCDMLPipeline.ipynb` and update the `DATA_PATH` variable in Cell 4 to point at an NDA-licensed ABCD data file. The main notebook's cell structure and parameter reference are documented in `README.md`.